# Causal forecast-error and grid-headroom features

This notebook builds the Issue #4 modelling table. Every join is evaluated as of the prediction issue time: future forecast revisions and observations are never allowed into a row.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from gridtoev.forecast_features import build_forecast_feature_table

## Load normalized sources

The inputs retain forecast publication times and completed half-hour observations. Those timestamps are the evidence used by the leakage checks.

In [ ]:
processed = ROOT / 'data' / 'processed'
asof = pd.read_csv(processed / 'forecast_features_asof_30_60.csv')
history = pd.read_csv(processed / 'eirgrid_core_history_30min.csv.gz')
vintages = pd.read_csv(processed / 'forecast_vintages.csv.gz', low_memory=False)
len(asof), len(history), len(vintages)

## Engineer and validate

The builder creates forecast net load, renewable share/surplus, an SNSP proxy, interconnector headroom/utilisation, revision disagreement, and causal rolling forecast errors. It fails closed if it detects future information, duplicate natural keys, misaligned horizons or infinite values.

In [ ]:
features, dictionary, quality = build_forecast_feature_table(asof, history, vintages)
quality

In [ ]:
key_features = [
    'issue_timestamp_utc', 'forecast_horizon_minutes',
    'forecast_net_load_mw', 'forecast_renewable_surplus_mw',
    'forecast_snsp_proxy_ratio', 'interconnector_export_headroom_mw',
    'wind_forecast_revision_delta_mw', 'wind_forecast_error_mae_2h',
    'latest_completed_state_stale_flag',
]
features[key_features].head()

## Write reproducible artifacts

Missing feature values are preserved and paired with explicit flags. The trained pipeline can then impute numeric inputs without pretending that unavailable data was observed.

In [ ]:
features.to_csv(processed / 'forecast_model_features_30_60.csv', index=False, date_format='%Y-%m-%dT%H:%M:%SZ')
dictionary.to_csv(processed / 'forecast_model_feature_dictionary.csv', index=False)
print(f'Wrote {len(features):,} rows and documented all {len(dictionary):,} columns.')